# Inkling 2 Mini: Linux token hashes on an sm_120 card (RTX PRO 6000)

Builds TensorFold at `ab268f923` (ThinkOffApp/TensorFold `local/macos-cuda-pr`, on ashhart/TensorFold main 043ee2bf6),
serves Inkling 2 Mini with no drafter and 16-bit settings, and records the reply token ids for TensorFold's 16 fixed
prompts. The same script runs against the RTX 5090 on the Mac; the two JSON files are then compared bit for bit.

Runtime: **Runtime > Change runtime type > GPU: RTX PRO 6000** (or another sm_120 card). Then **Runtime > Run all**.
It takes about 30 to 45 minutes, most of it the CUDA toolkit, the kernel build and the model download.
At the end it downloads `colab-hashes.json`: send that file to the room.

In [ ]:
# 1. The GPU: must be compute capability 12.0 (sm_120) with a CUDA 13 driver
import subprocess
q = subprocess.run(["nvidia-smi","--query-gpu=name,compute_cap,driver_version,memory.total","--format=csv,noheader"],
                   capture_output=True, text=True).stdout.strip()
print(q)
name, cap, drv, mem = [x.strip() for x in q.split(",")]
assert cap == "12.0", f"this GPU is compute capability {cap}; pick an sm_120 card (RTX PRO 6000) in Change runtime type"
assert int(drv.split(".")[0]) >= 580, f"driver {drv} is older than CUDA 13 needs (580+)"
print("OK: sm_120 card with a CUDA 13 driver")

In [ ]:
%%bash
# 2. CUDA 13.3 toolkit (nvcc), the version the Mac's kernel images were built with
set -e
. /etc/os-release
REPO=ubuntu${VERSION_ID/./}
cd /tmp
wget -q https://developer.download.nvidia.com/compute/cuda/repos/$REPO/x86_64/cuda-keyring_1.1-1_all.deb
dpkg -i cuda-keyring_1.1-1_all.deb > /dev/null
apt-get -qq update
apt-get -qq install -y cuda-toolkit-13-3 > /dev/null
/usr/local/cuda-13.3/bin/nvcc --version | tail -2

In [ ]:
%%bash
# 3. Zig 0.17.0 (TensorFold's build refuses any other version) and TensorFold at the pinned commit
set -e
cd /content
[ -x zig-x86_64-linux-0.17.0/zig ] || { wget -q https://ziglang.org/download/0.17.0/zig-x86_64-linux-0.17.0.tar.xz && tar xf zig-x86_64-linux-0.17.0.tar.xz; }
zig-x86_64-linux-0.17.0/zig version
[ -d TensorFold ] || git clone -q --filter=blob:none -b local/macos-cuda-pr https://github.com/ThinkOffApp/TensorFold.git
cd TensorFold && git checkout -q ab268f923 && git log --oneline -1

In [ ]:
%%bash
# 4. Kernel images for sm_120, then compare them with the 166 images the Mac uses
set -e
cd /content/TensorFold
export PATH=/content/zig-x86_64-linux-0.17.0:$PATH
mkdir -p out-fatbins
zig build fatbins -Dnvcc=/usr/local/cuda-13.3/bin/nvcc -Dsm=120 --prefix $PWD/out-fatbins 2>&1 | tail -3
ls out-fatbins/fatbin | wc -l
wget -q -O /content/mac.sha256 https://raw.githubusercontent.com/ThinkOffApp/mac-amd-cuda-llm-cluster/linux-compare/linux-compare/fatbins-sm120.sha256
cd out-fatbins/fatbin && sha256sum -c /content/mac.sha256 --quiet && echo "kernel images: all 166 byte-identical to the Mac's" || echo "SOME KERNEL IMAGES DIFFER from the Mac's (lines above); the token comparison is then not like for like" 

In [ ]:
%%bash
# 5. The server
set -e
cd /content/TensorFold
export PATH=/content/zig-x86_64-linux-0.17.0:$PATH
zig build -Dgpu=sm_120 -Dfatbins=$PWD/out-fatbins/fatbin 2>&1 | tail -3
ls -la zig-out/native/bin/tensorfold-native

In [ ]:
%%bash
# 6. The model, and its weight hashes (must match the Mac's)
set -e
pip -q install -U "huggingface_hub[cli]" > /dev/null
hf download thinkingmachines/Inkling-2-Mini > /dev/null
D=$(ls -d ~/.cache/huggingface/hub/models--thinkingmachines--Inkling-2-Mini/snapshots/* | head -1)
echo "snapshot: $D"
cd "$D"
sha256sum -L model-00001-of-00001.safetensors mtp.safetensors
echo "the Mac's:"
echo "65e60f51aa1284d4fb022584908c315270334d991828bb774af60584523539d1  model-00001-of-00001.safetensors"
echo "7c3b651317f01f73bcf4e1dceba66ef8b7e1a1eac6bc171e882e690a4b10a1d2  mtp.safetensors" 

In [ ]:
# 7. Serve: no drafts (not even the model's own MTP head), 16-bit settings (no --kv-cache fp8, no TF_INKLING2_* switches), context 8192
import subprocess, time, urllib.request, os
log = open("/content/serve.log", "w")
srv = subprocess.Popen(["/content/TensorFold/zig-out/native/bin/tensorfold-native", "serve", "thinkingmachines/Inkling-2-Mini",
                        "--host", "127.0.0.1", "--port", "8766", "--name", "inkling-2-mini", "--context", "8192",
                        "--no-drafts", "--no-update-check"], stdout=log, stderr=subprocess.STDOUT, cwd="/content/TensorFold")
for i in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8766/v1/models", timeout=2); print(f"server up after {2*i} s"); break
    except Exception:
        if srv.poll() is not None: break
        time.sleep(2)
else:
    pass
print(open("/content/serve.log").read()[-1500:])
assert srv.poll() is None, "the server exited; the log is above" 

In [ ]:
%%bash
# 8. Token hashes: 16 prompts, 128 tokens each, every prompt asked twice
set -e
cd /content
wget -q -O hash_run.py https://raw.githubusercontent.com/ThinkOffApp/mac-amd-cuda-llm-cluster/linux-compare/linux-compare/hash_run.py
wget -q -O bench_prompts.json https://raw.githubusercontent.com/ThinkOffApp/mac-amd-cuda-llm-cluster/linux-compare/linux-compare/bench_prompts.json
python3 hash_run.py --base http://127.0.0.1:8766 --label colab-sm120-linux --out colab-hashes.json

In [ ]:
# 9. Save the result: send colab-hashes.json to the room
from google.colab import files
files.download("/content/colab-hashes.json")